In [1]:
import re
import nltk
from collections import Counter
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

True

In [2]:
tweets = [
    "Just finished my breakfast!! Check out https://t.co/xyz #goodMorning ☀️😊",
    "Loving the new #Python3.10 features... performance is amazing!!!",
    "@john_doe The movie was REALLY bad... waste of time :(",
    "Don't miss our special offer!!! Visit our website: https://example.com",
    "I can't believe it's Friday!!! #TGIF #Weekend",
    "Machine Learning & Deep Learning are the future of tech!!! 🚀🚀🚀"
]

### Tâche 1 : Conversion et gestion de base
1. Convertissez tous les tweets en minuscules
2. Supprimez les URLs (identifiez-les avec une regex)
3. Supprimez les mentions (@user) et les hashtags
4. Supprimez les emojis
5. Remplacez les caractères dupliqués excessifs (!!!, ???, ...) par un seul exemplaire

In [3]:
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
MENTION_HASHTAG_PATTERN = re.compile(r"[@#]\w+")
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F300-\U0001F5FF"
    "\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F"
    "\U0001F780-\U0001F7FF"
    "\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FAFF"
    "\U00002700-\U000027BF"
    "\U000024C2-\U0001F251"
    "]+",
    flags=re.UNICODE,
)
DUPLICATED_PUNCT_PATTERN = re.compile(r"([!?\.])\1+")
PUNCT_WITHOUT_APOSTROPHE_PATTERN = re.compile(r"[^\w\s']")
MULTI_SPACE_PATTERN = re.compile(r"\s+")

stop_words = set(stopwords.words("english")).union(stopwords.words("french"))
lemmatizer = WordNetLemmatizer()


def basic_clean(text: str) -> str:
    text = text.lower()
    text = URL_PATTERN.sub(" ", text)
    text = MENTION_HASHTAG_PATTERN.sub(" ", text)
    text = EMOJI_PATTERN.sub(" ", text)
    text = DUPLICATED_PUNCT_PATTERN.sub(r"\1", text)
    return text


cleaned_basic = [basic_clean(tweet) for tweet in tweets]
print("=== Tâche 1 : sortie nettoyage de base ===")
for row in cleaned_basic:
    print(row)

=== Tâche 1 : sortie nettoyage de base ===
just finished my breakfast! check out      
loving the new  .10 features. performance is amazing!
  the movie was really bad. waste of time :(
don't miss our special offer! visit our website:  
i can't believe it's friday!    
machine learning & deep learning are the future of tech!  


### Tâche 2 : Gestion de la ponctuation et des caractères spéciaux
1. Supprimez la ponctuation inutile, SAUF les apostrophes qui doivent être conservées pour les contractions
2. Supprimez les espaces multiples
3. Gérez les cas particuliers :
   - "can't" doit rester "can't" (contraction)
   - "don't" doit rester "don't"
4. Testez votre logique sur : `"I can't believe... it's AMAZING!!!"`

In [4]:
def clean_punctuation(text: str) -> str:
    text = PUNCT_WITHOUT_APOSTROPHE_PATTERN.sub(" ", text)
    text = MULTI_SPACE_PATTERN.sub(" ", text).strip()
    return text


test_text = "I can't believe... it's AMAZING!!!"
print("=== Tâche 2 : test contractions ===")
print(clean_punctuation(basic_clean(test_text)))

cleaned_texts = [clean_punctuation(text) for text in cleaned_basic]
print("\n=== Tâche 2 : tweets normalisés ===")
for row in cleaned_texts:
    print(row)

=== Tâche 2 : test contractions ===
i can't believe it's amazing

=== Tâche 2 : tweets normalisés ===
just finished my breakfast check out
loving the new 10 features performance is amazing
the movie was really bad waste of time
don't miss our special offer visit our website
i can't believe it's friday
machine learning deep learning are the future of tech


### Tâche 3 : Tokenization et stopwords
1. Tokenisez le texte nettoyé (utilisez NLTK ou une approche simple split)
2. Créez une liste personnalisée de stopwords en français ET en anglais
3. Supprimez les stopwords
4. Gardez les contractions comme "can't", "don't" comme tokens entiers

In [5]:
def tokenize_and_remove_stopwords(text: str) -> list[str]:
    tokens = text.split()
    return [token for token in tokens if token not in stop_words]


tokens_per_tweet = [tokenize_and_remove_stopwords(text) for text in cleaned_texts]
print("=== Tâche 3 : tokens significatifs ===")
for tokens in tokens_per_tweet:
    print(tokens)

=== Tâche 3 : tokens significatifs ===
['finished', 'breakfast', 'check']
['loving', 'new', '10', 'features', 'performance', 'amazing']
['movie', 'really', 'bad', 'waste', 'time']
['miss', 'special', 'offer', 'visit', 'website']
["can't", 'believe', 'friday']
['machine', 'learning', 'deep', 'learning', 'future', 'tech']


### Tâche 4 : Lemmatisation
1. Implémentez la lemmatisation avec NLTK WordNetLemmatizer
2. Appliquez-la sur les tokens filtrés
3. Comparez avant/après (montrez des exemples)
4. Optionnel : Utilisez SpaCy pour un résultat amélioré


In [6]:
def lemmatize_tokens(tokens: list[str]) -> list[str]:
    return [lemmatizer.lemmatize(token, pos="v") for token in tokens]


lemmatized_per_tweet = [lemmatize_tokens(tokens) for tokens in tokens_per_tweet]
print("=== Tâche 4 : comparaison avant / après lemmatisation ===")
for index, (before, after) in enumerate(zip(tokens_per_tweet[:3], lemmatized_per_tweet[:3]), start=1):
    print(f"Tweet {index}")
    print("  Avant:", before)
    print("  Après:", after)

=== Tâche 4 : comparaison avant / après lemmatisation ===
Tweet 1
  Avant: ['finished', 'breakfast', 'check']
  Après: ['finish', 'breakfast', 'check']
Tweet 2
  Avant: ['loving', 'new', '10', 'features', 'performance', 'amazing']
  Après: ['love', 'new', '10', 'feature', 'performance', 'amaze']
Tweet 3
  Avant: ['movie', 'really', 'bad', 'waste', 'time']
  Après: ['movie', 'really', 'bad', 'waste', 'time']


### Tâche 5 : Pipeline complet
1. Créez une fonction `clean_tweet(text)` qui applique TOUTES les étapes précédentes
2. Testez-la sur l'ensemble du corpus
3. Affichez avant/après pour 3 tweets
4. Mesurez la réduction du vocabulaire (nombre de tokens uniques avant/après)

In [7]:
def clean_tweet(text: str) -> list[str]:
    normalized_text = clean_punctuation(basic_clean(text))
    filtered_tokens = tokenize_and_remove_stopwords(normalized_text)
    return lemmatize_tokens(filtered_tokens)


cleaned_corpus = [clean_tweet(tweet) for tweet in tweets]

print("=== Tâche 5 : avant / après (3 tweets) ===")
for index, original in enumerate(tweets[:3], start=1):
    print(f"\nTweet {index} - Original : {original}")
    print(f"Tweet {index} - Nettoyé  : {cleaned_corpus[index - 1]}")

raw_tokens = [token for tweet in tweets for token in tweet.lower().split()]
clean_tokens = [token for tweet_tokens in cleaned_corpus for token in tweet_tokens]

raw_vocab_size = len(set(raw_tokens))
clean_vocab_size = len(set(clean_tokens))

print("\n=== Réduction de vocabulaire ===")
print(f"Taille vocabulaire brut   : {raw_vocab_size}")
print(f"Taille vocabulaire nettoyé: {clean_vocab_size}")
print(f"Réduction                : {raw_vocab_size - clean_vocab_size}")

=== Tâche 5 : avant / après (3 tweets) ===

Tweet 1 - Original : Just finished my breakfast!! Check out https://t.co/xyz #goodMorning ☀️😊
Tweet 1 - Nettoyé  : ['finish', 'breakfast', 'check']

Tweet 2 - Original : Loving the new #Python3.10 features... performance is amazing!!!
Tweet 2 - Nettoyé  : ['love', 'new', '10', 'feature', 'performance', 'amaze']

Tweet 3 - Original : @john_doe The movie was REALLY bad... waste of time :(
Tweet 3 - Nettoyé  : ['movie', 'really', 'bad', 'waste', 'time']

=== Réduction de vocabulaire ===
Taille vocabulaire brut   : 49
Taille vocabulaire nettoyé: 27
Réduction                : 22
